# Technique card code (instructor reference)

The code for every technique card, to refer to while helping students. Not meant to be run top to bottom.

- Each code cell starts with the notebook section it belongs to.
- Lines ending in the card ID (e.g. `# A6`) are new or changed; `# removed:` lines are deleted.
- Preprocessing cells show only the new lines inside the `for df in [train, test]:` loop; other sections show the whole cell.
- *Builds on* lists the cards already applied when this code was tested (`test_cards.py`). A student's notebook may differ; the idea matters, not an exact match.

# A. Preparing the data

## A1. Use more columns

**Where:** Features.  
*Builds on:* A2

In [ ]:
# Section: Features
# removed: FEATURES = [
    # removed: "transaction_hour",
    # removed: "is_international",
    # removed: "failed_pin_attempts_24h",
    # removed: "txn_count_last_1h",
    # removed: "merchant_risk_score",
    # removed: "device_trust_score",
# removed: ]
FEATURES = train.select_dtypes("number").columns.drop("is_fraud").tolist()  # A1

X = train[FEATURES].fillna(0)
X_test = test[FEATURES].fillna(0)
y = train["is_fraud"]

## A2. Scale the inputs

**Where:** Right after the split.  
*Builds on:* the starter notebook

In [ ]:
# Section: Imports
from sklearn.preprocessing import StandardScaler  # A2

In [ ]:
# Section: Train / validation split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler().fit(X_train)  # A2
X_train = scaler.transform(X_train)  # A2
X_val = scaler.transform(X_val)  # A2
X_test = scaler.transform(X_test)  # A2

## A3. Log-transform skewed columns

**Where:** Preprocessing loop.  
*Builds on:* A2, A1

In [ ]:
# Section: Preprocessing
for df in [train, test]:
    ...  # your earlier preprocessing lines
    # removed: pass
    for col in ["amount_inr", "customer_avg_spend_90d", "card_age_days", "distance_from_home_km", "txn_count_last_1h"]:  # A3
        df[col] = np.log1p(df[col])  # A3

## A4. One-hot encode text columns

**Where:** Features, where `X` and `X_test` are built.  
*Builds on:* A2, A1, A3

In [ ]:
# Section: Features
FEATURES = train.select_dtypes("number").columns.drop("is_fraud").tolist()

# removed: X = train[FEATURES].fillna(0)
# removed: X_test = test[FEATURES].fillna(0)
TEXT = ["merchant_category", "channel", "card_type", "city_tier"]  # A4
X = pd.get_dummies(train[FEATURES + TEXT], columns=TEXT, dtype=float).fillna(0)  # A4
X_test = pd.get_dummies(test[FEATURES + TEXT], columns=TEXT, dtype=float).fillna(0)  # A4
X_test = X_test.reindex(columns=X.columns, fill_value=0)  # A4
y = train["is_fraud"]

## A5. Label-encode text columns

**Where:** Features.  
*Builds on:* the full solution (every card in the guide's path)

In [ ]:
# Section: Features
FEATURES = train.select_dtypes("number").columns.drop("is_fraud").tolist()
FEATURES = [col for col in FEATURES if col not in ["batch_number", "acquirer_code", "pos_software_version"]]

TEXT = ["merchant_category", "channel", "card_type", "city_tier"]
# removed: X = pd.get_dummies(train[FEATURES + TEXT], columns=TEXT, dtype=float)
# removed: X_test = pd.get_dummies(test[FEATURES + TEXT], columns=TEXT, dtype=float)
# removed: X_test = X_test.reindex(columns=X.columns, fill_value=0)
X = train[FEATURES + TEXT].copy()  # A5
X_test = test[FEATURES + TEXT].copy()  # A5
for col in TEXT:  # A5
    categories = sorted(train[col].unique())  # A5
    X[col] = pd.Categorical(X[col], categories=categories).codes  # A5
    X_test[col] = pd.Categorical(X_test[col], categories=categories).codes  # A5
y = train["is_fraud"]

## A6. Fill blanks with the median

**Where:** Right after the split, before any scaling.  
*Builds on:* A2, A1, A3, A4

In [ ]:
# Section: Features
FEATURES = train.select_dtypes("number").columns.drop("is_fraud").tolist()

TEXT = ["merchant_category", "channel", "card_type", "city_tier"]
# removed: X = pd.get_dummies(train[FEATURES + TEXT], columns=TEXT, dtype=float).fillna(0)
# removed: X_test = pd.get_dummies(test[FEATURES + TEXT], columns=TEXT, dtype=float).fillna(0)
X = pd.get_dummies(train[FEATURES + TEXT], columns=TEXT, dtype=float)  # A6
X_test = pd.get_dummies(test[FEATURES + TEXT], columns=TEXT, dtype=float)  # A6
X_test = X_test.reindex(columns=X.columns, fill_value=0)
y = train["is_fraud"]

In [ ]:
# Section: Train / validation split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

medians = X_train.median()  # A6
X_train = X_train.fillna(medians)  # A6
X_val = X_val.fillna(medians)  # A6
X_test = X_test.fillna(medians)  # A6
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

## A7. Add "was missing" flags

**Where:** Preprocessing loop, after the other column changes (and before any filling).  
*Builds on:* A2, A1, A3, A4, A6

In [ ]:
# Section: Preprocessing
for df in [train, test]:
    ...  # your earlier preprocessing lines
    for col in train.columns[train.isna().any()]:  # A7
        df[col + "_missing"] = df[col].isna().astype(int)  # A7

## A8. Handle special codes

**Where:** Preprocessing loop, before any log.  
*Builds on:* A2, A1, A3, A4, A6, A7

In [ ]:
# Section: Preprocessing
for df in [train, test]:
    ...  # your earlier preprocessing lines
    df["never_chargeback"] = (df["days_since_last_chargeback"] == -1).astype(int)  # A8
    df["days_since_last_chargeback"] = np.log1p(df["days_since_last_chargeback"].replace(-1, np.nan))  # A8

## A9. Drop columns that cannot carry signal

**Where:** Features.  
*Builds on:* A2, A1, A3, A4, A6, A7, A8, B1, B2, C1, D1

In [ ]:
# Section: Features
FEATURES = train.select_dtypes("number").columns.drop("is_fraud").tolist()
FEATURES = [col for col in FEATURES if col not in ["batch_number", "acquirer_code", "pos_software_version"]]  # A9

TEXT = ["merchant_category", "channel", "card_type", "city_tier"]
X = pd.get_dummies(train[FEATURES + TEXT], columns=TEXT, dtype=float)
X_test = pd.get_dummies(test[FEATURES + TEXT], columns=TEXT, dtype=float)
X_test = X_test.reindex(columns=X.columns, fill_value=0)
y = train["is_fraud"]

# B. Training settings

## B1. Change the optimizer or learning rate

**Where:** Model, in `compile`.  
*Builds on:* A2, A1, A3, A4, A6, A7, A8

In [ ]:
# Section: Model
keras.utils.set_random_seed(0)

model = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1],)),
    keras.layers.Dense(4, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),
])
# removed: model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.01), loss="binary_crossentropy", metrics=["accuracy"])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss="binary_crossentropy", metrics=["accuracy"])  # B1
model.summary()

## B2. Train for more epochs

**Where:** Training.  
*Builds on:* A2, A1, A3, A4, A6, A7, A8, B1

In [ ]:
# Section: Training
history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                    # removed: epochs=10, batch_size=32, verbose=2)
                    epochs=100, batch_size=32, verbose=2)  # B2

## B3. Change the batch size

**Where:** Training.  
*Builds on:* the full solution (every card in the guide's path)

In [ ]:
# Section: Training
reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5)
history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                    # removed: epochs=100, batch_size=32, callbacks=[reduce_lr], verbose=2)
                    epochs=100, batch_size=256, callbacks=[reduce_lr], verbose=2)  # B3

## B4. Lower the learning rate when progress stalls

**Where:** Training.  
*Builds on:* A2, A1, A3, A4, A6, A7, A8, B1, B2, C1, D1, A9, E1, E2, E3

In [ ]:
# Section: Training
reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5)  # B4
history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                    # removed: epochs=100, batch_size=32, verbose=2)
                    epochs=100, batch_size=32, callbacks=[reduce_lr], verbose=2)  # B4

# C. Model capacity

## C1. A wider or deeper network

**Where:** Model.  
*Builds on:* A2, A1, A3, A4, A6, A7, A8, B1, B2

In [ ]:
# Section: Model
keras.utils.set_random_seed(0)

model = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1],)),
    # removed: keras.layers.Dense(4, activation="relu"),
    keras.layers.Dense(128, activation="relu"),  # C1
    keras.layers.Dense(128, activation="relu"),  # C1
    keras.layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

## C2. A very deep network

**Where:** Model.  
*Builds on:* the full solution (every card in the guide's path)

In [ ]:
# Section: Model
keras.utils.set_random_seed(0)

model = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1],)),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(128, activation="relu"),  # C2
    keras.layers.Dropout(0.3),  # C2
    keras.layers.Dense(128, activation="relu"),  # C2
    keras.layers.Dropout(0.3),  # C2
    keras.layers.Dense(128, activation="relu"),  # C2
    keras.layers.Dropout(0.3),  # C2
    keras.layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

## C3. A different activation

**Where:** Model.  
*Builds on:* the full solution (every card in the guide's path)

In [ ]:
# Section: Model
keras.utils.set_random_seed(0)

model = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1],)),
    # removed: keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(128, activation="sigmoid"),  # C3
    keras.layers.Dropout(0.3),
    # removed: keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(128, activation="sigmoid"),  # C3
    keras.layers.Dropout(0.3),
    keras.layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

## C4. Batch normalization

**Where:** Model.  
*Builds on:* the full solution (every card in the guide's path)

In [ ]:
# Section: Model
keras.utils.set_random_seed(0)

model = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1],)),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.BatchNormalization(),  # C4
    keras.layers.Dropout(0.3),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.BatchNormalization(),  # C4
    keras.layers.Dropout(0.3),
    keras.layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

# D. Regularisation

## D1. Dropout

**Where:** Model.  
*Builds on:* A2, A1, A3, A4, A6, A7, A8, B1, B2, C1

In [ ]:
# Section: Model
keras.utils.set_random_seed(0)

model = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1],)),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.3),  # D1
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.3),  # D1
    keras.layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

## D2. L2 penalty

**Where:** Model.  
*Builds on:* A2, A1, A3, A4, A6, A7, A8, B1, B2, C1

In [ ]:
# Section: Model
keras.utils.set_random_seed(0)

model = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1],)),
    # removed: keras.layers.Dense(128, activation="relu"),
    # removed: keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(128, activation="relu", kernel_regularizer=keras.regularizers.L2(1e-3)),  # D2
    keras.layers.Dense(128, activation="relu", kernel_regularizer=keras.regularizers.L2(1e-3)),  # D2
    keras.layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

## D3. L1 penalty

**Where:** Model.  
*Builds on:* the full solution (every card in the guide's path)

In [ ]:
# Section: Model
keras.utils.set_random_seed(0)

model = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1],)),
    # removed: keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(128, activation="relu", kernel_regularizer=keras.regularizers.L1(1e-4)),  # D3
    keras.layers.Dropout(0.3),
    # removed: keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(128, activation="relu", kernel_regularizer=keras.regularizers.L1(1e-4)),  # D3
    keras.layers.Dropout(0.3),
    keras.layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

## D4. Early stopping

**Where:** Training.  
*Builds on:* the full solution (every card in the guide's path)

In [ ]:
# Section: Training
early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)  # D4
reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5)
history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                    # removed: epochs=100, batch_size=32, callbacks=[reduce_lr], verbose=2)
                    epochs=100, batch_size=32, callbacks=[early_stop, reduce_lr], verbose=2)  # D4

# E. Feature engineering

## E1. Compare the amount with the customer's usual spend

**Where:** Preprocessing loop, before any log of the amount columns (it needs raw rupees).  
*Builds on:* A2, A1, A3, A4, A6, A7, A8, B1, B2, C1, D1, A9

In [ ]:
# Section: Preprocessing
for df in [train, test]:
    ...  # your earlier preprocessing lines
    df["amount_vs_usual"] = np.log(df["amount_inr"] / df["customer_avg_spend_90d"])  # E1

## E2. Treat the hour as a circle

**Where:** Preprocessing loop.  
*Builds on:* A2, A1, A3, A4, A6, A7, A8, B1, B2, C1, D1, A9, E1

In [ ]:
# Section: Preprocessing
for df in [train, test]:
    ...  # your earlier preprocessing lines
    df["hour_sin"] = np.sin(2 * np.pi * df["transaction_hour"] / 24)  # E2
    df["hour_cos"] = np.cos(2 * np.pi * df["transaction_hour"] / 24)  # E2

## E3. Look at the amounts themselves

**Where:** Preprocessing loop, before any log of `amount_inr`.  
*Builds on:* A2, A1, A3, A4, A6, A7, A8, B1, B2, C1, D1, A9, E1, E2

In [ ]:
# Section: Preprocessing
for df in [train, test]:
    ...  # your earlier preprocessing lines
    df["amount_is_round"] = (df["amount_inr"] % 500 == 0).astype(int)  # E3

# F. Final polish

## F1. Average several models

**Where:** Submission, replacing the prediction line.  
*Builds on:* the full solution (every card in the guide's path)

In [ ]:
# Section: Submission
# removed: pred = (model.predict(X_test) > 0.5).astype(int).ravel()
probs = []  # F1
for seed in [0, 1, 2]:  # F1
    keras.utils.set_random_seed(seed)  # F1

    model = keras.Sequential([  # F1
        keras.layers.Input(shape=(X_train.shape[1],)),  # F1
        keras.layers.Dense(128, activation="relu"),  # F1
        keras.layers.Dropout(0.3),  # F1
        keras.layers.Dense(128, activation="relu"),  # F1
        keras.layers.Dropout(0.3),  # F1
        keras.layers.Dense(1, activation="sigmoid"),  # F1
    ])  # F1
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss="binary_crossentropy", metrics=["accuracy"])  # F1
    reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5)  # F1
    model.fit(X_train, y_train, validation_data=(X_val, y_val),  # F1
              epochs=100, batch_size=32, callbacks=[reduce_lr], verbose=0)  # F1
    probs.append(model.predict(X_test).ravel())  # F1
pred = (np.mean(probs, axis=0) > 0.5).astype(int)  # F1
submission = pd.DataFrame({"transaction_id": test["transaction_id"], "is_fraud": pred})
submission.to_csv("submission.csv", index=False)